**Exemplary load profiles**  
This notebook prepares exemplary demand and generation profiles for the first prototype of Keks.  
The load profiles include:  
- Kritis (Fireforce) from data shared by Stadt Offenburg
- Residential house from the online-tool nPro  
- Industry from the online-tool nPro  

The generation profiles include:  
- per unit values of solar generation based on solar irradiation provided by pvgis  
- Heat pump COP time series, based solely on approximation and author-experience.  

@Author: AqibThenndan

In [195]:
import os
import pandas as pd
import numpy as np

In [196]:
os.getcwd()

'c:\\Keks-repo\\KEKS'

In [197]:
import src.utils as utils

In [198]:
dir = os.path.join(os.getcwd(), 'data')

In [199]:
#DF to save load profiles of all nodes in proto-network
load_profiles_pu = pd.DataFrame({})

#DF with peak load powers
load_p_max = pd.DataFrame({}, columns = ['P_max(kW)'])

gen_p_max = pd.DataFrame({}, columns = ['P_max(kW)'])

gen_profiles_pu = pd.DataFrame({})

In [200]:
full_year = pd.date_range(start = '2026-01-01 00:00:00', end = '2026-12-31 23:00:00', freq = 'h')

# Load profiles

## Kritis
Using dataset shared by yamit as source, easier that way

In [201]:
sog = pd.read_excel(os.path.join(dir,"Stadt_Offenburg_Lastkurven_Stündlich_Detailed.xlsx" ), index_col = 0, sheet_name = 'Feuerwehr', parse_dates = True)
sog.index = sog.index.str.replace(r"\.\d+$", "", regex=True) #one timestamp had microseconds wierdly
sog.index = pd.to_datetime(sog.index)
sog.index = sog.index.map(lambda t: t.replace(year=2026))
sog

,Stromverbrauch (kWh),BHKW (kWh),Netzlieferung (kWh),Bezug (kWh),Wärmeverbrauch (kWh),Kessel (kWh),BHKW (kWh).1,Gas BHKW (kWh),Gas BHKW (m3),Gas Kessel (kWh),...,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33
Datum Uhrzeit,,,,,,,,,,,,,,,,,,,,,
2026-01-01 01:00:00,5.65503,0.0,0.0,5.65503,14.0,14.0,0.0,0.0,0.0,14.893617,...,NaN,NaN,NaN,Kessel,NaN,NaN,NaN,Gesamt Strom,10450.034389,kWh
2026-01-01 02:00:00,5.05503,0.0,0.0,5.05503,12.0,12.0,0.0,0.0,0.0,12.765957,...,4.7,kW,NaN,Model,Vaillant ecoTEC plus VC 5-5,NaN,NaN,Gesamt Wärme,30487.920000,kWh
2026-01-01 03:00:00,4.65503,0.0,0.0,4.65503,19.0,19.0,0.0,0.0,0.0,20.212766,...,12.5,kW,NaN,Wirkungsgrad,94,%,NaN,NaN,NaN,NaN
2026-01-01 04:00:00,5.45503,0.0,0.0,5.45503,18.0,18.0,0.0,0.0,0.0,19.148936,...,"Vaillant ecoPower 4,7",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-01-01 05:00:00,5.25503,0.0,0.0,5.25503,16.0,16.0,0.0,0.0,0.0,17.021277,...,90,%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-12-31 19:59:59,1.43270,0.0,0.0,1.43270,10.0,10.0,0.0,0.0,0.0,10.638298,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-12-31 20:59:59,1.43270,0.0,0.0,1.43270,10.0,10.0,0.0,0.0,0.0,10.638298,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-12-31 21:59:59,1.43270,0.0,0.0,1.43270,11.0,11.0,0.0,0.0,0.0,11.702128,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [202]:
sog = utils.clean_index(snaps = full_year, df = sog)

{Timestamp('2026-10-03 21:00:00'), Timestamp('2026-04-13 04:00:00'), Timestamp('2026-12-20 13:00:00'), Timestamp('2026-08-16 22:00:00'), Timestamp('2026-11-04 10:00:00'), Timestamp('2026-10-22 12:00:00'), Timestamp('2026-10-04 21:00:00'), Timestamp('2026-11-18 08:00:00'), Timestamp('2026-07-26 19:00:00'), Timestamp('2026-08-12 14:00:00'), Timestamp('2026-09-09 14:00:00'), Timestamp('2026-09-26 08:00:00'), Timestamp('2026-09-02 06:00:00'), Timestamp('2026-06-29 19:00:00'), Timestamp('2026-06-19 16:00:00'), Timestamp('2026-05-17 07:00:00'), Timestamp('2026-06-14 00:00:00'), Timestamp('2026-11-27 02:00:00'), Timestamp('2026-06-10 04:00:00'), Timestamp('2026-11-12 21:00:00'), Timestamp('2026-09-20 02:00:00'), Timestamp('2026-05-21 10:00:00'), Timestamp('2026-07-06 02:00:00'), Timestamp('2026-09-06 11:00:00'), Timestamp('2026-09-26 14:00:00'), Timestamp('2026-05-10 13:00:00'), Timestamp('2026-09-10 19:00:00'), Timestamp('2026-07-24 22:00:00'), Timestamp('2026-07-02 16:00:00'), Timestamp('20

In [203]:
sog.isna().sum()

Stromverbrauch (kWh)       0
BHKW (kWh)                 0
Netzlieferung (kWh)        0
Bezug (kWh)                0
Wärmeverbrauch (kWh)       0
Kessel (kWh)               0
BHKW (kWh).1               0
Gas BHKW (kWh)             0
Gas BHKW (m3)              0
Gas Kessel (kWh)           0
Gas Kessel (m3)            0
Gas Gesamt (kWh)           0
Gas Gesamt (m3)            0
Nutzungstyp                0
Month                      0
Jahr                       0
Jahreszeit                 0
Tag                        0
Unnamed: 19             8760
Unnamed: 20             8755
Unnamed: 21             8755
Unnamed: 22             8760
Unnamed: 23             8755
Unnamed: 24             8756
Unnamed: 25             8757
Unnamed: 26             8760
Unnamed: 27             8757
Unnamed: 28             8758
Unnamed: 29             8759
Unnamed: 30             8760
Unnamed: 31             8758
Unnamed: 32             8758
Unnamed: 33             8758
dtype: int64

In [204]:
# capacities of the BHKW and Kessel
gen_p_max.loc['Kritis_chp', 'P_max(kW)'] = np.round(sog['BHKW (kWh).1'].max(), 2)
gen_p_max.loc['Kritis_kessel', 'P_max(kW)'] = np.round(sog['Kessel (kWh)'].max(), 2)

gen_p_max

,P_max(kW)
Kritis_chp,15.6
Kritis_kessel,41.88


In [205]:
load_p_max.loc['kritis_thermal', 'P_max(kW)'] = np.round(sog['Wärmeverbrauch (kWh)'].max(), 2)
load_profiles_pu['kritis_thermal'] = sog['Wärmeverbrauch (kWh)']/load_p_max.loc['kritis_thermal', 'P_max(kW)']

load_p_max.loc['kritis_electric', 'P_max(kW)'] = np.round(sog['Stromverbrauch (kWh)'].max(), 2)
load_profiles_pu['kritis_electric'] = sog['Stromverbrauch (kWh)']/load_p_max.loc['kritis_electric', 'P_max(kW)']

In [206]:
sog.index[0],sog.index[-1]

(Timestamp('2026-01-01 00:00:00'), Timestamp('2026-12-31 23:00:00'))

In [207]:
load_profiles_pu

,kritis_thermal,kritis_electric
2026-01-01 00:00:00,0.334288,0.933173
2026-01-01 01:00:00,0.286533,0.834163
2026-01-01 02:00:00,0.453677,0.768157
2026-01-01 03:00:00,0.429799,0.900170
2026-01-01 04:00:00,0.382044,0.867167
...,...,...
2026-12-31 19:00:00,0.238777,0.236419
2026-12-31 20:00:00,0.238777,0.236419
2026-12-31 21:00:00,0.262655,0.236419
2026-12-31 22:00:00,0.310411,0.236419


In [208]:
load_profiles_pu.isna().sum()

kritis_thermal     0
kritis_electric    0
dtype: int64

## Residential

10 MFHs aggregated  
12 aparments each, 120 apartments in total  
Heated floor area/avg/apartment = 100 sq.m
total heated floor area = 12.000 sq.m  

npro:
SH - 248 kWh/m2/yr  
DHW - 21 kWh/m2/yr  
El - 22 kWh/m2/yr  
 



In [209]:
import locale

locale.setlocale(locale.LC_TIME, "en_US.UTF-8")

residential = pd.read_excel(os.path.join(dir, "proto_residential.xlsx"), skiprows = 16, usecols = [1,2,3], index_col = 0)
residential_el = pd.read_excel(os.path.join(dir, "proto_residential.xlsx"), sheet_name= 'Electricity', skiprows = 16, usecols = [1,2,3], index_col = 0)

residential.index = pd.to_datetime(residential.index, format = '%a, %d.%m. %H:%M')
residential.index = residential.index.map(lambda t: t.replace(year=2026))

residential_el.index = pd.to_datetime(residential_el.index, format = '%a, %d.%m. %H:%M')
residential_el.index = residential_el.index.map(lambda t: t.replace(year=2026))

residential['Total'] = residential.sum(axis=1)
residential

,Space heating (kW),Domestic hot water (kW),Total
Time,,,
2026-01-01 00:00:00,482.13,14.00,496.13
2026-01-01 01:00:00,453.04,8.24,461.28
2026-01-01 02:00:00,420.21,4.39,424.60
2026-01-01 03:00:00,456.78,1.98,458.76
2026-01-01 04:00:00,487.74,4.39,492.13
...,...,...,...
2026-12-31 19:00:00,1040.80,45.09,1085.89
2026-12-31 20:00:00,1089.00,44.45,1133.45
2026-12-31 21:00:00,993.56,35.47,1029.03


In [210]:
len(residential.index)/24

365.0

In [211]:
residential.index

DatetimeIndex(['2026-01-01 00:00:00', '2026-01-01 01:00:00',
               '2026-01-01 02:00:00', '2026-01-01 03:00:00',
               '2026-01-01 04:00:00', '2026-01-01 05:00:00',
               '2026-01-01 06:00:00', '2026-01-01 07:00:00',
               '2026-01-01 08:00:00', '2026-01-01 09:00:00',
               ...
               '2026-12-31 14:00:00', '2026-12-31 15:00:00',
               '2026-12-31 16:00:00', '2026-12-31 17:00:00',
               '2026-12-31 18:00:00', '2026-12-31 19:00:00',
               '2026-12-31 20:00:00', '2026-12-31 21:00:00',
               '2026-12-31 22:00:00', '2026-12-31 23:00:00'],
              dtype='datetime64[us]', name='Time', length=8760, freq=None)

In [212]:

residential_el = utils.clean_index(snaps = full_year, df = residential_el)
residential = utils.clean_index(snaps = full_year, df = residential)

In [213]:
residential.isna().sum()

Space heating (kW)         0
Domestic hot water (kW)    0
Total                      0
dtype: int64

In [214]:
load_p_max.loc['residential_thermal', 'P_max(kW)'] = np.round(residential['Total'].max(), 2)
load_profiles_pu['residential_thermal'] = residential['Total']/load_p_max.loc['residential_thermal', 'P_max(kW)']

load_p_max.loc['residential_electric', 'P_max(kW)'] = np.round(residential_el['Plug loads (kW)'].max(), 2)
load_profiles_pu['residential_electric'] = residential_el['Plug loads (kW)']/load_p_max.loc['residential_electric', 'P_max(kW)']

In [215]:
load_profiles_pu['residential_thermal'].isna().sum()

np.int64(0)

In [216]:
residential.isna().sum(), residential_el.isna().sum()

(Space heating (kW)         0
 Domestic hot water (kW)    0
 Total                      0
 dtype: int64,
 Plug loads (kW)    0
 E-mobility (kW)    0
 dtype: int64)

In [217]:
load_profiles_pu[load_profiles_pu['residential_electric'].isna()]

,kritis_thermal,kritis_electric,residential_thermal,residential_electric


In [218]:
residential_el.loc['2026-04-10 10:00:00']/load_p_max.loc['residential_electric', 'P_max(kW)']

Plug loads (kW)    0.510721
E-mobility (kW)    0.000000
Name: 2026-04-10 10:00:00, dtype: float64

In [219]:
load_profiles_pu.isna().sum()

kritis_thermal          0
kritis_electric         0
residential_thermal     0
residential_electric    0
dtype: int64

In [220]:
load_p_max

,P_max(kW)
kritis_thermal,41.88
kritis_electric,6.06
residential_thermal,2111.29
residential_electric,66.69


### District heating  

Adding two more buses, which would recieve heating supplied a DH network.  
Residential2, & Residential3  
Residential2 aggregates 3 MFHs, while Residential3 aggregates 5MFHs.
The demand profile would be the same as the residential node, the peak load will be adjusted accordinlgly here.  

In [221]:
type(load_p_max)

pandas.DataFrame

In [222]:
resid2 = load_p_max.loc[['residential_thermal', 'residential_electric']].copy(deep = True)
resid3 = resid2.copy(deep = True)
resid2['P_max(kW)'] *= 3/10 
resid2.index = ['residential2_thermal', 'residential2']
resid3['P_max(kW)'] *= 5/10
resid3.index = ['residential3_thermal', 'residential3']

load_p_max = pd.concat([load_p_max, resid2, resid3], axis = 0)
# load_p_max

## Industry
npro:  
5000 m2  
Old construction  
sh: 74 kwh/m2/yr
dhw: 2 kWh/m2/yr  
el: 72 kWh/m2/yr

In [223]:
locale.setlocale(locale.LC_TIME, "de_DE.UTF-8")

industry = pd.read_excel(os.path.join(dir, "industry_proto.xlsx"), skiprows = 16, usecols = [1,2,3], index_col = 0)

industry.index = pd.to_datetime(industry.index, format = '%a, %d.%m. %H:%M')
industry.index = industry.index.map(lambda t: t.replace(year=2026))
industry['Total'] = industry.sum(axis=1)
# industry
industry_el = pd.read_excel(os.path.join(dir, "industry_proto.xlsx"), sheet_name = "Strom",  skiprows = 16, usecols = [1,2,3], index_col = 0)
industry_el.index = pd.to_datetime(industry_el.index, format = '%a, %d.%m. %H:%M')
industry_el.index = industry_el.index.map(lambda t: t.replace(year=2026))

industry_el

,Nutzerstrom (kW),Elektromobilität (kW)
Zeit,,
2026-01-01 00:00:00,34.07,0
2026-01-01 01:00:00,35.09,0
2026-01-01 02:00:00,32.03,0
2026-01-01 03:00:00,33.73,0
2026-01-01 04:00:00,32.51,0
...,...,...
2026-12-31 19:00:00,37.38,0
2026-12-31 20:00:00,36.47,0
2026-12-31 21:00:00,32.85,0


In [224]:
industry = utils.clean_index(snaps = full_year, df = industry)
industry_el = utils.clean_index(snaps = full_year, df = industry_el)

In [225]:
load_p_max.loc['industry_thermal', 'P_max(kW)'] = np.round(industry['Total'].max(), 2)
load_profiles_pu['industry_thermal'] = industry['Total']/load_p_max.loc['industry_thermal', 'P_max(kW)']

load_p_max.loc['industry_electric', 'P_max(kW)'] = np.round(industry_el['Nutzerstrom (kW)'].max(), 2)
load_profiles_pu['industry_electric'] = industry_el['Nutzerstrom (kW)']/load_p_max.loc['industry_electric', 'P_max(kW)']

# PV PU generation profile

In [226]:
import pvlib
import pandas as pd
import numpy as np
import os
import json
from pvlib.pvsystem import PVSystem, Array, FixedMount
from pvlib.modelchain import ModelChain
from pvlib.location import Location

In [227]:
def fetch_pvgis(lat, long, year, array_configs):
    '''
    Fetches PVGIS hourly poa data for each array, and returns a list of dataframes.
    Additionally, prepares the df with necessary columns and resamples to 15 min with interpolation.
    '''
    weather_list = []
    for arr in array_configs:
        data, input = pvlib.iotools.get_pvgis_hourly(
            latitude=lat,
            longitude=long,
            start=year,
            end=year,
            components=True,
            pvcalculation=False,
            outputformat='csv',
            surface_tilt=arr['tilt'],
            surface_azimuth=180 - arr['azimuth']  # PVGIS: 0=south, -90=east, +90=west
        )

        weather = data[['poa_direct', 'poa_sky_diffuse', 'poa_ground_diffuse',
                        'temp_air', 'wind_speed']].copy()

        weather['poa_diffuse'] = weather['poa_sky_diffuse'] + weather['poa_ground_diffuse']
        weather['poa_global']  = weather['poa_direct'] + weather['poa_diffuse']

        weather = weather[['poa_global', 'poa_direct', 'poa_diffuse', 'temp_air', 'wind_speed']]
        # weather = weather.resample('15T', offset='10min').interpolate() #there's a 10 min offset in the timestamps of pvgis data
        weather.index = weather.index - pd.Timedelta(minutes=10)
        weather_list.append(weather)

    return weather_list

In [228]:

#The arrays of pv modules, mutliple possible
arrays_ = [{'tilt' : 30,
            'azimuth' : 180}]
latitude, longitude = 48.45, 7.95


module_info = ['SandiaMod','SunPower_128_Cell_Module__2009__E__']
inverter_info = ['cecinverter', 'AEconversion_GMbH__INV500_90US_xxxxx__208V_']

modules_db = pvlib.pvsystem.retrieve_sam(module_info[0])
module = modules_db[module_info[1]] #replace BAD_CHARS = ' -.()[]:+/",' ; with simply _
# retreiving inverter from db
sapm_inverters = pvlib.pvsystem.retrieve_sam(inverter_info[0])
inverter = sapm_inverters[inverter_info[1]]
#temperature model
temperature_model_parameters = pvlib.temperature.TEMPERATURE_MODEL_PARAMETERS['sapm']['open_rack_glass_glass']

#creates array objects for all array configs, and appends to list arrays
arrays = []
for arr in arrays_:
    arrays.append(Array(
        mount=FixedMount(surface_tilt=arr['tilt'], surface_azimuth=arr['azimuth']), 
        module_parameters=module,
        temperature_model_parameters=temperature_model_parameters,
        modules_per_string=1,
        strings=1
    ))

location = Location(
    latitude,
    longitude,
    name="Stegermatt",
    altitude=110,
    tz="Etc/GMT-1",

)

#getting irradiance data from PVGIS
weather_list = fetch_pvgis(latitude, longitude, 2023, arrays_) #only data till 2023 available
weather = pd.DataFrame(index=weather_list[0].index) #empty df with correct index, required with fetch pvgis

system = PVSystem(arrays=arrays, inverter_parameters=inverter)
mc = ModelChain(system, location)

mc.run_model_from_poa(weather_list)

pv_gen = mc.results.ac
pv_gen.index = pv_gen.index.map(lambda t: t.replace(year=2026)) #replace year to 2026, as the other profiles are for 2026
pv_gen = pv_gen.clip(lower = 0)
# pu profile
## divided by 500 W here, the installed capacity, but max is much lower than this; this needs to be looked into
pv_gen = pv_gen/500 #pu profile

In [229]:
gen_profiles_pu['pv'] = pv_gen

# HP cop

In [230]:
365/4

91.25

In [231]:
winter_day = [
    2.5, 2.4, 2.4, 2.3, 2.3, 2.4,
    2.5, 2.6, 2.8, 3.0, 3.2, 3.3,
    3.4, 3.5, 3.5, 3.4, 3.2, 3.0,
    2.9, 2.8, 2.7, 2.6, 2.5, 2.5
]

summer_day = [
    4.2, 4.1, 4.1, 4.0, 4.0, 4.1,
    4.3, 4.5, 4.7, 4.9, 5.1, 5.2,
    5.3, 5.4, 5.4, 5.3, 5.2, 5.0,
    4.8, 4.7, 4.5, 4.4, 4.3, 4.2
]
spring_day = [
    3.3, 3.2, 3.2, 3.1, 3.1, 3.2,
    3.4, 3.6, 3.8, 4.0, 4.2, 4.3,
    4.4, 4.5, 4.5, 4.4, 4.3, 4.1,
    3.9, 3.8, 3.7, 3.5, 3.4, 3.3
]
autumn_day = [
    3.1, 3.0, 3.0, 2.9, 2.9, 3.0,
    3.2, 3.4, 3.6, 3.8, 4.0, 4.1,
    4.2, 4.3, 4.3, 4.2, 4.0, 3.8,
    3.6, 3.5, 3.4, 3.3, 3.2, 3.1
]

cop_profile = (
    winter_day * 62 +
    spring_day * 92 +
    summer_day * 91 +
    autumn_day * 91 +
    winter_day * 29
)
gen_profiles_pu['hp_cop'] = cop_profile


In [232]:
len(cop_profile)

8760

# Exporting data

## Generation data

In [233]:
gen_p_max

,P_max(kW)
Kritis_chp,15.6
Kritis_kessel,41.88


In [234]:
gen_profiles_pu

,pv,hp_cop
time,,
2026-01-01 00:00:00+00:00,0.0,2.5
2026-01-01 01:00:00+00:00,0.0,2.4
2026-01-01 02:00:00+00:00,0.0,2.4
2026-01-01 03:00:00+00:00,0.0,2.3
2026-01-01 04:00:00+00:00,0.0,2.3
...,...,...
2026-12-31 19:00:00+00:00,0.0,2.8
2026-12-31 20:00:00+00:00,0.0,2.7
2026-12-31 21:00:00+00:00,0.0,2.6


In [235]:
# gen_profiles = pd.read_excel(os.path.join(dir, "gen_profiles_pu.xlsx"), index_col = 0)
# gen_profiles
# gen_profiles.loc[:, 'Roof_pv'] = gen_profiles_pu['pv']
# gen_profiles.loc[:, 'HP_COP'] = gen_profiles_pu['hp_cop']
gen_profiles_pu = gen_profiles_pu.rename(columns = {
    'time': 'snapshot',
    'pv':'Roof_pv',
    'hp_cop':'HP_COP'
})
gen_profiles_pu.index.name = 'snapshot'
gen_profiles_pu.index = gen_profiles_pu.index.tz_localize(None)
gen_profiles_pu.to_excel(os.path.join(dir, "gen_profiles_pu.xlsx"))

## Loads

In [236]:
gen_profiles_pu.index[0], gen_profiles_pu.index[-1], len(gen_profiles_pu.index)

(Timestamp('2026-01-01 00:00:00'), Timestamp('2026-12-31 23:00:00'), 8760)

In [237]:
load_profiles_pu.index = load_profiles_pu.index.round('h')
load_profiles_pu.index = load_profiles_pu.index - pd.Timedelta(hours=1) #the load profiles are shifted by 1 hour, so the first value is for 2026-01-01 01:00:00, not 2026-01-01 00:00:00

In [238]:
load_profiles_pu.index[0], load_profiles_pu.index[-1], len(load_profiles_pu.index)

(Timestamp('2025-12-31 23:00:00'), Timestamp('2026-12-31 22:00:00'), 8760)

In [239]:
pd.date_range('2026-01-01 00:00:00', '2026-12-31 23:00:00', freq = 'h')

DatetimeIndex(['2026-01-01 00:00:00', '2026-01-01 01:00:00',
               '2026-01-01 02:00:00', '2026-01-01 03:00:00',
               '2026-01-01 04:00:00', '2026-01-01 05:00:00',
               '2026-01-01 06:00:00', '2026-01-01 07:00:00',
               '2026-01-01 08:00:00', '2026-01-01 09:00:00',
               ...
               '2026-12-31 14:00:00', '2026-12-31 15:00:00',
               '2026-12-31 16:00:00', '2026-12-31 17:00:00',
               '2026-12-31 18:00:00', '2026-12-31 19:00:00',
               '2026-12-31 20:00:00', '2026-12-31 21:00:00',
               '2026-12-31 22:00:00', '2026-12-31 23:00:00'],
              dtype='datetime64[us]', length=8760, freq='h')

In [240]:
load_profiles_pu.isna().sum()

kritis_thermal          0
kritis_electric         0
residential_thermal     0
residential_electric    0
industry_thermal        0
industry_electric       0
dtype: int64

In [241]:
load_profiles_pu.index.name = 'snapshot'
load_profiles_pu = load_profiles_pu.rename(columns = {
    'kritis_electric' : 'kritis',
    'industry_electric' : 'industry',
    'residential_electric' : 'residential',
})

#correcting the order
loads = pd.read_excel(os.path.join(dir, 'loads.xlsx'), index_col = 0)
order = loads.index
load_profiles_pu = load_profiles_pu[order]

#setting the peak load
load_p_max.rename(index = {'residential_electric': 'residential',
                           'kritis_electric': 'kritis',
                           'industry_electric': 'industry'}, inplace = True)
loads['p_max'] = load_p_max['P_max(kW)']/1000 #in MW

load_profiles_pu.to_excel(os.path.join(dir, "load_profiles_pu.xlsx"))
loads.to_excel(os.path.join(dir, "loads.xlsx"))

In [242]:
loads

,bus,carrier,p_max
name,,,
industry,Industry_1,NaN,0.05696
kritis,Kritis_1,NaN,0.00606
residential,Residential_1,NaN,0.06669
industry_thermal,Industry_1_thermal,NaN,0.17511
kritis_thermal,Kritis_1_thermal,NaN,0.04188
residential_thermal,Residential_1_thermal,NaN,2.11129


In [243]:
load_profiles_pu.columns

Index(['industry', 'kritis', 'residential', 'industry_thermal',
       'kritis_thermal', 'residential_thermal'],
      dtype='str')

In [244]:
order

Index(['industry', 'kritis', 'residential', 'industry_thermal',
       'kritis_thermal', 'residential_thermal'],
      dtype='str', name='name')